# MLB Stats API Demo

Using the [python-mlb-statsapi](https://github.com/zero-sum-seattle/python-mlb-statsapi) library to access MLB data.

Install with:
```
pip install python-mlb-statsapi
```

In [2]:
import mlbstatsapi
import pandas as pd

mlb = mlbstatsapi.Mlb()

## 1. All MLB Teams (sport_id=1)

In [3]:
# sport_id=1 is Major League Baseball
mlb_teams = mlb.get_teams(sport_id=1)

print(f"Total MLB teams: {len(mlb_teams)}\n")

mlb_teams_data = [
    {
        "id": team.id,
        "name": team.name,
        "abbreviation": team.abbreviation,
        "league": team.league.name if team.league else None,
        "division": team.division.name if team.division else None,
        "venue": team.venue.name if team.venue else None,
        "location": team.location_name,
    }
    for team in mlb_teams
]

df_mlb = pd.DataFrame(mlb_teams_data).sort_values("name").reset_index(drop=True)
df_mlb

Total MLB teams: 30



,id,name,abbreviation,league,division,venue,location
0,109,Arizona Diamondbacks,AZ,National League,National League West,Chase Field,Phoenix
1,133,Athletics,ATH,American League,American League West,Sutter Health Park,Sacramento
2,144,Atlanta Braves,ATL,National League,National League East,Truist Park,Atlanta
3,110,Baltimore Orioles,BAL,American League,American League East,Oriole Park at Camden Yards,Baltimore
4,111,Boston Red Sox,BOS,American League,American League East,Fenway Park,Boston
5,112,Chicago Cubs,CHC,National League,National League Central,Wrigley Field,Chicago
6,145,Chicago White Sox,CWS,American League,American League Central,Rate Field,Chicago
7,113,Cincinnati Reds,CIN,National League,National League Central,Great American Ball Park,Cincinnati
8,114,Cleveland Guardians,CLE,American League,American League Central,Progressive Field,Cleveland
9,115,Colorado Rockies,COL,National League,National League West,Coors Field,Denver


## 2. Available Sports / Leagues

Use `get_sports()` to discover sport IDs for minor leagues.

In [4]:
sports = mlb.get_sports()

sports_data = [
    {"id": sport.id, "name": sport.name, "abbreviation": getattr(sport, 'abbreviation', None)}
    for sport in sports
]

df_sports = pd.DataFrame(sports_data).sort_values("id").reset_index(drop=True)
df_sports

,id,name,abbreviation
0,1,Major League Baseball,MLB
1,11,Triple-A,AAA
2,12,Double-A,AA
3,13,High-A,A+
4,14,Single-A,A
5,16,Rookie,ROK
6,17,Winter Leagues,WIN
7,21,Minor League Baseball,Minors
8,22,College Baseball,College
9,23,Independent Leagues,IND


## 3. All Minor League Teams

Minor league sport IDs:
- `11` = Triple-A
- `12` = Double-A
- `13` = High-A
- `14` = Single-A
- `16` = Rookie

We fetch teams for each level and combine them.

In [5]:
MINOR_LEAGUE_SPORT_IDS = {
    11: "Triple-A",
    # 12: "Double-A",
    # 13: "High-A",
    # 14: "Single-A",
    # 16: "Rookie",
}

minor_league_rows = []

for sport_id, level_name in MINOR_LEAGUE_SPORT_IDS.items():
    teams = mlb.get_teams(sport_id=sport_id)
    print(f"{level_name} (sport_id={sport_id}): {len(teams)} teams")
    for team in teams:
        minor_league_rows.append({
            "level": level_name,
            "sport_id": sport_id,
            "id": team.id,
            "name": team.name,
            "abbreviation": team.abbreviation,
            "league": team.league.name if team.league else None,
            "location": team.location_name,
            "venue": team.venue.name if team.venue else None,
        })

df_minor = pd.DataFrame(minor_league_rows).sort_values(["sport_id", "name"]).reset_index(drop=True)
print(f"\nTotal minor league teams: {len(df_minor)}")
df_minor

Triple-A (sport_id=11): 30 teams

Total minor league teams: 30


,level,sport_id,id,name,abbreviation,league,location,venue
0,Triple-A,11,342,Albuquerque Isotopes,ABQ,Pacific Coast League,Albuquerque,Isotopes Park
1,Triple-A,11,422,Buffalo Bisons,BUF,International League,Buffalo,Sahlen Field
2,Triple-A,11,494,Charlotte Knights,CLT,International League,Charlotte,Truist Field
3,Triple-A,11,445,Columbus Clippers,COL,International League,Columbus,Huntington Park
4,Triple-A,11,234,Durham Bulls,DUR,International League,Durham,Durham Bulls Athletic Park
5,Triple-A,11,4904,El Paso Chihuahuas,ELP,Pacific Coast League,El Paso,Southwest University Park
6,Triple-A,11,431,Gwinnett Stripers,GWN,International League,Lawrenceville,Gwinnett Field
7,Triple-A,11,484,Indianapolis Indians,IND,International League,Indianapolis,Victory Field
8,Triple-A,11,451,Iowa Cubs,IOW,International League,Des Moines,Principal Park
9,Triple-A,11,564,Jacksonville Jumbo Shrimp,JAX,International League,Jacksonville,Vystar Ballpark


## 4. Summary

## 5. Active MLB Players

In [6]:

players = mlb.get_people(sport_id=1)

players_data = [
    {
        "id": p.id,
        "full_name": p.full_name,
        "first_name": p.first_name,
        "last_name": p.last_name,
        "jersey_number": getattr(p, "primary_number", None),
        "position": p.primary_position.abbreviation if p.primary_position else None,
        "position_name": p.primary_position.name if p.primary_position else None,
        "team": p.current_team.name if p.current_team else None,
        "bat_side": p.bat_side.description if p.bat_side else None,
        "pitch_hand": p.pitch_hand.description if p.pitch_hand else None,
        "birth_date": getattr(p, "birth_date", None),
        "active": getattr(p, "active", None),
    }
    for p in players
]

df_players = pd.DataFrame(players_data).sort_values(["team", "last_name"]).reset_index(drop=True)
print(f"Total players: {len(df_players)}")
df_players

Total players: 1199


,id,full_name,first_name,last_name,jersey_number,position,position_name,team,bat_side,pitch_hand,birth_date,active
0,671096,Andrew Abbott,Andrew,Abbott,41,P,Pitcher,None,Left,Left,1999-06-01,True
1,690953,Mick Abel,McLean,Abel,20,P,Pitcher,None,Right,Right,2001-08-18,True
2,691769,Philip Abner,Philip,Abner,50,P,Pitcher,None,Left,Left,2002-05-05,True
3,682928,CJ Abrams,Paul,Abrams,5,SS,Shortstop,None,Left,Right,2000-10-03,True
4,650556,Bryan Abreu,Bryan,Abreu,52,P,Pitcher,None,Right,Right,1997-04-22,True
...,...,...,...,...,...,...,...,...,...,...,...,...
1194,800018,Chen Zhong-Ao Zhuang,Chen,Zhuang,73,P,Pitcher,None,Right,Right,2000-08-25,True
1195,686632,Steven Zobac,Steven,Zobac,63,P,Pitcher,None,Left,Right,2000-10-14,True
1196,691172,Yosver Zulueta,Yosver,Zulueta,61,P,Pitcher,None,Right,Right,1998-01-23,True
1197,518595,Travis d'Arnaud,Travis,d'Arnaud,25,C,Catcher,None,Right,Right,1989-02-10,True


## 6. Team Roster

In [7]:
ROSTER_TEAM_ID = 139        # e.g. 147 = Yankees; see df_mlb for IDs
ROSTER_TYPE   = "fullRoster"    # 40Man | fullSeason | fullRoster | active | etc.

roster = mlb.get_team_roster(team_id=ROSTER_TEAM_ID, rosterType=ROSTER_TYPE)

roster_data = [
    {
        "id": p.id,
        "full_name": p.full_name,
        "jersey_number": p.jersey_number,
        "position": p.primary_position.abbreviation if p.primary_position else None,
        "position_name": p.primary_position.name if p.primary_position else None,
        "status": p.status.description if p.status else None,
        "bat_side": p.bat_side.description if p.bat_side else None,
        "pitch_hand": p.pitch_hand.description if p.pitch_hand else None,
        "note": getattr(p, "note", None),
    }
    for p in roster
]

df_roster = pd.DataFrame(roster_data).sort_values("position").reset_index(drop=True)
print(f"Team ID {ROSTER_TEAM_ID} · {ROSTER_TYPE} roster: {len(df_roster)} players")
df_roster

Team ID 139 · fullRoster roster: 213 players


,id,full_name,jersey_number,position,position_name,status,bat_side,pitch_hand,note
0,700178,Ryan McCoy,,1B,First Base,Active,None,None,None
1,650490,Yandy Díaz,2,1B,First Base,Active,None,None,None
2,666018,Jonathan Aranda,8,1B,First Base,Active,None,None,None
3,829403,Trace Phillips,,1B,First Base,Active,None,None,None
4,687545,Blake Robertson,31,1B,First Base,Active,None,None,None
...,...,...,...,...,...,...,...,...,...
208,813922,Jack Lines,17,SS,Shortstop,Active,None,None,None
209,821782,Andreimi Antunez,23,SS,Shortstop,Active,None,None,None
210,670764,Taylor Walls,6,SS,Shortstop,Active,None,None,None
211,836590,Victor Valdez,,SS,Shortstop,Active,None,None,None


## 7. Schedule

In [16]:
SCHEDULE_START_DATE = "2026-01-01"
SCHEDULE_END_DATE   = "2026-12-31"
# No team_id filter — fetch all MLB games

schedule = mlb.get_schedule(
    start_date=SCHEDULE_START_DATE,
    end_date=SCHEDULE_END_DATE,
    sport_id=1,
)

schedule_rows = []
if schedule:
    for day in schedule.dates:
        for game in day.games:
            away = game.teams.away if game.teams else None
            home = game.teams.home if game.teams else None
            schedule_rows.append({
                "game_pk":            game.game_pk,
                "game_date":          game.game_date,
                "date":               day.date,
                "day_night":          getattr(game, "day_night", None),
                "away_team":          away.team.name if away and away.team else None,
                "home_team":          home.team.name if home and home.team else None,
                "away_score":         away.score if away else None,
                "home_score":         home.score if home else None,
                "status":             game.status.detailed_state if game.status else None,
                "venue":              game.venue.name if game.venue else None,
                "venue_id":           game.venue.id   if game.venue else None,
                "series_description": getattr(game, "series_description", None),
                "game_number":        getattr(game, "series_game_number", None),
                "games_in_series":    getattr(game, "games_in_series", None),
            })

df_schedule = pd.DataFrame(schedule_rows)
print(f"Games from {SCHEDULE_START_DATE} to {SCHEDULE_END_DATE}: {len(df_schedule)}")
df_schedule

Games from 2026-01-01 to 2026-12-31: 2920


,game_pk,game_date,date,day_night,away_team,home_team,away_score,home_score,status,venue,venue_id,series_description,game_number,games_in_series
0,832034,2026-02-20T18:05:00Z,2026-02-20,day,New York Yankees,Baltimore Orioles,0.0,2.0,Final,Ed Smith Stadium,2508,Spring Training,1.0,1.0
1,832033,2026-02-20T18:05:00Z,2026-02-20,day,Northeastern Huskies,Boston Red Sox,3.0,18.0,Final,JetBlue Park,4309,Exhibition Game,NaN,NaN
2,831829,2026-02-20T20:05:00Z,2026-02-20,day,Kansas City Royals,Texas Rangers,7.0,3.0,Final,Surprise Stadium,2603,Spring Training,1.0,1.0
3,832042,2026-02-20T20:05:00Z,2026-02-20,day,Chicago White Sox,Chicago Cubs,8.0,1.0,Final,Sloan Park,4629,Spring Training,1.0,1.0
4,831959,2026-02-20T20:10:00Z,2026-02-20,day,Arizona Diamondbacks,Colorado Rockies,3.0,2.0,Final,Salt River Fields at Talking Stick,4249,Spring Training,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2915,823650,2026-09-27T19:10:00Z,2026-09-27,day,Texas Rangers,Minnesota Twins,NaN,NaN,Scheduled,Target Field,3312,Regular Season,3.0,3.0
2916,824542,2026-09-27T19:10:00Z,2026-09-27,day,Colorado Rockies,Chicago White Sox,NaN,NaN,Scheduled,Rate Field,4,Regular Season,3.0,3.0
2917,823814,2026-09-27T19:10:00Z,2026-09-27,day,Atlanta Braves,Miami Marlins,NaN,NaN,Scheduled,loanDepot park,4169,Regular Season,3.0,3.0
2918,823731,2026-09-27T19:10:00Z,2026-09-27,day,St. Louis Cardinals,Milwaukee Brewers,NaN,NaN,Scheduled,American Family Field,32,Regular Season,3.0,3.0


In [17]:
schedule_csv_path = "mlb_schedule_2026.csv"
df_schedule[["game_date", "game_pk", "home_team", "away_team", "venue", "venue_id"]].to_csv(
    schedule_csv_path, index=False
)
print(f"Wrote {len(df_schedule)} rows to {schedule_csv_path}")

Wrote 2920 rows to mlb_schedule_2026.csv


## 8. Game Detail (GUMBO Feed)

In [9]:
GAME_PK = 776918  # use a game_pk from df_schedule above

game = mlb.get_game(game_id=GAME_PK)

# --- Game summary ---
gd   = game.game_data
ld   = game.live_data
info = gd.game_info if gd else None

away_team = gd.teams.away.name if gd and gd.teams and gd.teams.away else None
home_team = gd.teams.home.name if gd and gd.teams and gd.teams.home else None

summary = {
    "game_pk":          game.game_pk,
    "date":             gd.datetime.official_date if gd and gd.datetime else None,
    "time":             f"{gd.datetime.time} {gd.datetime.ampm}" if gd and gd.datetime else None,
    "status":           gd.status.detailed_state if gd and gd.status else None,
    "away_team":        away_team,
    "home_team":        home_team,
    "venue":            gd.venue.name if gd and gd.venue else None,
    "weather":          f"{gd.weather.temp}°F, {gd.weather.condition}" if gd and gd.weather else None,
    "wind":             gd.weather.wind if gd and gd.weather else None,
    "attendance":       info.attendance if info else None,
    "duration_min":     info.game_duration_minutes if info else None,
    "season":           gd.game.season if gd and gd.game else None,
}

print("=== Game Summary ===")
for k, v in summary.items():
    print(f"  {k:<16} {v}")

# --- Linescore ---
ls = ld.linescore if ld else None
if ls and ls.innings:
    inning_rows = []
    for inn in ls.innings:
        inning_rows.append({
            "inning":      inn.num,
            "away_runs":   getattr(inn.away, "runs",  None) if inn.away else None,
            "away_hits":   getattr(inn.away, "hits",  None) if inn.away else None,
            "away_errors": getattr(inn.away, "errors",None) if inn.away else None,
            "home_runs":   getattr(inn.home, "runs",  None) if inn.home else None,
            "home_hits":   getattr(inn.home, "hits",  None) if inn.home else None,
            "home_errors": getattr(inn.home, "errors",None) if inn.home else None,
        })
    df_linescore = pd.DataFrame(inning_rows)

    # totals row
    totals = pd.DataFrame([{
        "inning":      "TOTAL",
        "away_runs":   getattr(ls.teams.away, "runs",   None) if ls.teams and ls.teams.away else None,
        "away_hits":   getattr(ls.teams.away, "hits",   None) if ls.teams and ls.teams.away else None,
        "away_errors": getattr(ls.teams.away, "errors", None) if ls.teams and ls.teams.away else None,
        "home_runs":   getattr(ls.teams.home, "runs",   None) if ls.teams and ls.teams.home else None,
        "home_hits":   getattr(ls.teams.home, "hits",   None) if ls.teams and ls.teams.home else None,
        "home_errors": getattr(ls.teams.home, "errors", None) if ls.teams and ls.teams.home else None,
    }])
    df_linescore = pd.concat([df_linescore, totals], ignore_index=True)

    print(f"\n=== Linescore  ({away_team} @ {home_team}) ===")
    display(df_linescore)

# --- Decisions ---
decisions = getattr(ld, "decisions", None) if ld else None
if decisions:
    print("=== Decisions ===")
    for role in ("winner", "loser", "save"):
        p = getattr(decisions, role, None)
        if p:
            print(f"  {role.capitalize():<8} {p.full_name}")

=== Game Summary ===
  game_pk          776918
  date             2025-08-01
  time             7:35 PM
  status           Final
  away_team        Los Angeles Dodgers
  home_team        Tampa Bay Rays
  venue            George M. Steinbrenner Field
  weather          89°F, Partly Cloudy
  wind             4 mph, L To R
  attendance       10046
  duration_min     142
  season           2025

=== Linescore  (Los Angeles Dodgers @ Tampa Bay Rays) ===


,inning,away_runs,away_hits,away_errors,home_runs,home_hits,home_errors
0,1,2,2,0,0,1,0
1,2,0,0,0,0,2,0
2,3,0,1,0,0,1,0
3,4,2,4,0,0,0,0
4,5,1,1,0,0,0,0
5,6,0,1,1,0,1,0
6,7,0,0,0,0,0,0
7,8,0,0,0,0,0,0
8,9,0,0,0,0,1,0
9,TOTAL,5,9,1,0,6,0


=== Decisions ===
  Winner   Clayton Kershaw
  Loser    Shane Baz
  Save     Justin Wrobleski


## 9. Transactions

In [10]:
import requests

TRANSACTION_START_DATE = "2026-01-01"
TRANSACTION_END_DATE   = "2026-02-01"
TRANSACTION_TEAM_ID    = 139   # e.g. 147 for Yankees, or None for all teams
TRANSACTION_PLAYER_ID  = None   # e.g. 592450 for Gerrit Cole, or None for all players

params = {
    "startDate": TRANSACTION_START_DATE,
    "endDate":   TRANSACTION_END_DATE,
}
if TRANSACTION_TEAM_ID:
    params["teamId"] = TRANSACTION_TEAM_ID
if TRANSACTION_PLAYER_ID:
    params["playerId"] = TRANSACTION_PLAYER_ID

resp = requests.get("https://statsapi.mlb.com/api/v1/transactions", params=params)
resp.raise_for_status()
raw = resp.json().get("transactions", [])

transaction_rows = [
    {
        "id":               t.get("id"),
        "date":             t.get("date"),
        "effective_date":   t.get("effectiveDate"),
        "resolution_date":  t.get("resolutionDate"),
        "type_code":        t.get("typeCode"),
        "type_desc":        t.get("typeDesc"),
        "player_id":        t["person"]["id"]       if t.get("person")   else None,
        "player":           t["person"]["fullName"]  if t.get("person")   else None,
        "from_team":        t["fromTeam"]["name"]    if t.get("fromTeam") else None,
        "to_team":          t["toTeam"]["name"]      if t.get("toTeam")   else None,
        "description":      t.get("description"),
    }
    for t in raw
]

df_transactions = pd.DataFrame(transaction_rows).sort_values(["date", "player"]).reset_index(drop=True)
print(f"Transactions from {TRANSACTION_START_DATE} to {TRANSACTION_END_DATE}: {len(df_transactions)}")
df_transactions

Transactions from 2026-01-01 to 2026-02-01: 51


,id,date,effective_date,resolution_date,type_code,type_desc,player_id,player,from_team,to_team,description
0,880669,2026-01-06,2026-01-06,2026-01-06,SFA,Signed as Free Agent,658668.0,Edward Olivares,NaN,Tampa Bay Rays,Tampa Bay Rays signed free agent RF Edward Oli...
1,883247,2026-01-06,2026-01-06,2026-01-06,SC,Status Change,658668.0,Edward Olivares,NaN,Tampa Bay Rays,RF Edward Olivares roster status changed by Ta...
2,880169,2026-01-06,2026-01-06,NaN,TR,Trade,669234.0,Justyn-Henry Malloy,Detroit Tigers,Tampa Bay Rays,Detroit Tigers traded RF Justyn-Henry Malloy t...
3,880169,2026-01-06,2026-01-06,NaN,TR,Trade,NaN,NaN,Tampa Bay Rays,Detroit Tigers,Detroit Tigers traded RF Justyn-Henry Malloy t...
4,880993,2026-01-07,2026-01-07,2026-01-07,SFA,Signed as Free Agent,686678.0,Chase Solesky,NaN,Tampa Bay Rays,Tampa Bay Rays signed free agent RHP Chase Sol...
5,883235,2026-01-07,2026-01-07,2026-01-07,SC,Status Change,686678.0,Chase Solesky,NaN,Tampa Bay Rays,RHP Chase Solesky roster status changed by Tam...
6,880319,2026-01-07,2026-01-07,NaN,CLW,Claimed Off Waivers,691907.0,Tsung-Che Cheng,Pittsburgh Pirates,Tampa Bay Rays,Tampa Bay Rays claimed SS Tsung-Che Cheng off ...
7,880722,2026-01-08,2026-01-08,2026-01-08,SFA,Signed as Free Agent,666165.0,Blake Sabol,NaN,Tampa Bay Rays,Tampa Bay Rays signed free agent C Blake Sabol...
8,883246,2026-01-08,2026-01-08,2026-01-08,SC,Status Change,666165.0,Blake Sabol,NaN,Tampa Bay Rays,C Blake Sabol roster status changed by Tampa B...
9,884404,2026-01-08,2026-01-08,2026-01-08,SFA,Signed as Free Agent,657611.0,Cam Hill,NaN,Tampa Bay Rays,Tampa Bay Rays signed free agent RHP Cam Hill ...


## 10. Game Content (Highlights & Recap)

In [11]:
CONTENT_GAME_PK = 776918  # use a game_pk from df_schedule above

resp = requests.get(f"https://statsapi.mlb.com/api/v1/game/{CONTENT_GAME_PK}/content")
resp.raise_for_status()
content = resp.json()

# --- Recap article ---
recap = content.get("editorial", {}).get("recap", {}).get("mlb", {})
if recap:
    print("=== Recap ===")
    print(f"  Headline : {recap.get('headline')}")
    print(f"  Date     : {recap.get('date')}")
    print(f"  Blurb    : {recap.get('blurb')}")
    print()

# --- Highlights ---
items = content.get("highlights", {}).get("highlights", {}).get("items", [])

highlight_rows = []
for item in items:
    if item.get("type") != "video":
        continue
    playbacks = {p["name"]: p["url"] for p in item.get("playbacks", [])}
    highlight_rows.append({
        "id":          item.get("id"),
        "headline":    item.get("headline"),
        "description": item.get("description"),
        "duration":    item.get("duration"),
        "date":        item.get("date"),
        "mp4":         playbacks.get("mp4Avc"),
        "hls":         playbacks.get("hlsCloud"),
    })
    print(playbacks.get("hlsCloud"))

df_highlights = pd.DataFrame(highlight_rows)
print(f"=== Highlights ({len(df_highlights)} videos) ===")
df_highlights

=== Recap ===
  Headline : Freeman looking like old self as big as any Dodgers Deadline acquisition
  Date     : 2025-08-02T04:25:00Z
  Blurb    : TAMPA -- After nearly two months of frustration, Freddie Freeman is beginning to look much more like his normal self.

https://mlb-cuts-diamond.mlb.com/FORGE/2025/2025-08/01/3f12b062-8f2269f9-5a238e9e-csvm-diamondgcp-asset.m3u8
https://mlb-cuts-diamond.mlb.com/FORGE/2025/2025-08/01/6701779d-d627a2d0-ede16ce5-csvm-diamondgcp-asset.m3u8
https://mlb-cuts-diamond.mlb.com/FORGE/2025/2025-08/01/187940cb-6870d206-2c965be4-csvm-diamondgcp-asset.m3u8
https://mlb-cuts-diamond.mlb.com/FORGE/2025/2025-08/01/d334f126-fd5bcb33-d536a8cd-csvm-diamondgcp-asset.m3u8
https://mlb-cuts-diamond.mlb.com/FORGE/2025/2025-08/01/20446d14-05ca8641-15b53836-csvm-diamondgcp-asset.m3u8
https://mlb-cuts-diamond.mlb.com/FORGE/2025/2025-08/01/59697802-dcde022d-476de63e-csvm-diamondgcp-asset.m3u8
https://mlb-cuts-diamond.mlb.com/FORGE/2025/2025-08/01/7993acea-d089e91c-5bd62f4

,id,headline,description,duration,date,mp4,hls
0,dodgers-vs-rays-highlights-x3049,Dodgers vs. Rays Highlights,Freddie Freeman and the Dodgers take on Shane ...,00:03:02,2025-08-02T03:53:32.17Z,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...
1,condensed-game-lad-tb-8-1-25,Condensed Game: LAD@TB - 8/1/25,Condensed Game: Freddie Freeman and the Dodger...,00:08:46,2025-08-02T03:46:33.243Z,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...
2,freddie-freeman-homers-12-on-a-fly-ball-to-rig...,Freddie Freeman's solo homer (12),Freddie Freeman hits a solo home run to right ...,00:00:22,2025-08-02T00:53:13.706Z,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...
3,freddie-freeman-doubles-28-on-a-sharp-line-dri...,Freddie Freeman's two-run double,Freddie Freeman doubles to right field in the ...,00:00:20,2025-08-01T23:43:04.54Z,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...
4,shane-baz-in-play-run-s-to-alex-freeland,Alex Freeland's first career RBI,Alex Freeland singles to center field in the t...,00:00:15,2025-08-02T00:38:38.597Z,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...
5,shane-baz-s-eight-strikeouts,Shane Baz's eight strikeouts,Shane Baz tosses eight strikeouts through 5 in...,00:00:58,2025-08-02T01:25:19.185Z,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...
6,shane-baz-in-play-no-out-to-shohei-ohtani,Shohei Ohtani singles after a broken bat,Shohei Ohtani singles on a ground ball to seco...,00:00:24,2025-08-02T00:15:48.999Z,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...
7,clayton-kershaw-s-three-strikeouts,Clayton Kershaw's three strikeouts,Clayton Kershaw tosses three strikeouts throug...,00:00:44,2025-08-02T01:39:02.97Z,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...
8,shane-baz-makes-a-sweet-snag,Shane Baz makes a sweet snag,Shane Baz snags a line drive off Andy Pages' b...,00:00:16,2025-08-01T23:48:19.904Z,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...
9,shane-baz-in-play-run-s-to-mookie-betts,Mookie Betts' sacrifice fly,Mookie Betts hits a sacrifice fly to left fiel...,00:00:16,2025-08-02T00:40:06.4Z,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...,https://mlb-cuts-diamond.mlb.com/FORGE/2025/20...


## 11. Venues

In [12]:
SPORT_LEVELS = {
    1:  "MLB",
    11: "Triple-A",
    12: "Double-A",
    13: "High-A",
    14: "Single-A",
    16: "Rookie",
}

BASE_URL = "https://statsapi.mlb.com"

# Step 1: build venue_id → (team, sport) from teams endpoints
venue_team_map = {}
for sport_id, level_name in SPORT_LEVELS.items():
    teams = mlb.get_teams(sport_id=sport_id)
    for t in teams:
        vid = t.venue.id if t.venue else None
        if vid:
            venue_team_map[vid] = {"team": t.name, "sport": level_name}

# Step 2: fetch venues with hydrate=location,fieldInfo per sport level
venue_rows = []
seen_ids = set()
for sport_id, level_name in SPORT_LEVELS.items():
    resp = requests.get(
        f"{BASE_URL}/api/v1/venues",
        params={"sportId": sport_id, "hydrate": "location,fieldInfo"},
    )
    resp.raise_for_status()
    for v in resp.json().get("venues", []):
        vid = v.get("id")
        if vid in seen_ids:
            continue
        seen_ids.add(vid)
        coords = v.get("location", {}).get("defaultCoordinates", {})
        info   = venue_team_map.get(vid, {})
        venue_rows.append({
            "id":        vid,
            "name":      v.get("name"),
            "latitude":  coords.get("latitude"),
            "longitude": coords.get("longitude"),
            "team":      info.get("team"),
            "sport":     info.get("sport"),
            "capacity":  v.get("fieldInfo", {}).get("capacity"),
            "elevation": v.get("location", {}).get("elevation"),
        })

df_venues = (
    pd.DataFrame(venue_rows)
    .sort_values(["sport", "name"])
    .reset_index(drop=True)
)
print(f"Total venues: {len(df_venues)}")
df_venues

Total venues: 230


,id,name,latitude,longitude,team,sport,capacity,elevation
0,2740,7 17 Credit Union Park,41.077390,-81.521850,Akron RubberDucks,Double-A,9297.0,NaN
1,3329,Arvest Ballpark,36.159680,-94.194624,Northwest Arkansas Naturals,Double-A,7500.0,NaN
2,4329,Blue Wahoos Stadium,30.404841,-87.218835,Pensacola Blue Wahoos,Double-A,5038.0,NaN
3,6136,Covenant Health Park,35.972485,-83.914950,Knoxville Smokies,Double-A,7500.0,NaN
4,2779,Delta Dental Park,43.656240,-70.278730,Portland Sea Dogs,Double-A,6868.0,NaN
...,...,...,...,...,...,...,...,...
225,6115,Sloan Park Complex,NaN,NaN,NaN,NaN,NaN,NaN
226,2603,Surprise Stadium,33.627620,-112.378490,NaN,NaN,10714.0,NaN
227,2500,Tempe Diablo Stadium,33.400920,-111.970700,NaN,NaN,9808.0,NaN
228,2853,The Diamond,37.571749,-77.463391,NaN,NaN,12134.0,NaN


In [13]:
csv_path = "venues.csv"
df_venues.to_csv(csv_path, index=False)
print(f"Wrote {len(df_venues)} rows to {csv_path}")

Wrote 230 rows to venues.csv


In [14]:
print("=== Team Count Summary ===")
print(f"MLB teams:               {len(df_mlb)}")
print(f"Minor league teams:      {len(df_minor)}")
print()
print("Minor league breakdown:")
print(df_minor.groupby("level")["id"].count().rename("team_count").to_string())

=== Team Count Summary ===
MLB teams:               30
Minor league teams:      30

Minor league breakdown:
level
Triple-A    30
